# Module 2 — Player Scouting & Auction Intelligence

**Notebook 07** — Builds the complete scouting engine with BOTH batting and bowling DNA.

**Outputs:**
- `data/scouting_profiles.csv` — all players with batting DNA + bowling DNA + scouting scores
- `data/dna_matrix_scaled.npy` — normalized DNA matrix for cosine similarity
- `data/player_names_index.npy` — player name ordering for the matrix

## Cell 1 — Load data

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import process, fuzz
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv('../data/master_ipl.csv')
player_dna = pd.read_csv('../data/player_dna.csv')
player_career = pd.read_csv('../data/player_career_info.csv')
auction_2025 = pd.read_csv('../data/auction/ipl_2025_auction_players.csv')
auction_2026 = pd.read_csv('../data/auction/IPL_Mini_Auction_2026.csv')

master['date'] = pd.to_datetime(master['date'])
player_dna['date'] = pd.to_datetime(player_dna['date'])

print(f"Master: {len(master):,} deliveries")
print(f"Player DNA: {player_dna['batsman'].nunique()} batters")
print(f"Career info: {len(player_career)} players")

Master: 278,205 deliveries
Player DNA: 704 batters
Career info: 760 players


## Cell 2 — Build latest batting DNA per player

In [3]:
latest_dna = player_dna.sort_values('date').groupby('batsman').last().reset_index()

BATTING_DNA = [
    'career_avg', 'career_sr', 'avg_last5', 'sr_last5',
    'avg_last10', 'sr_last10', 'form_score', 'form_sr',
    'consistency', 'peak_score', 'boundary_rate',
    'avg_runs_Death', 'avg_runs_Middle', 'avg_runs_Powerplay',
    'avg_sr_Death', 'avg_sr_Middle', 'avg_sr_Powerplay',
    'pace_sr', 'spin_sr', 'spin_weakness', 'pace_weakness',
    'high_pressure_avg'
]

profiles = latest_dna[['batsman', 'date', 'matches_played'] + BATTING_DNA].copy()
profiles = profiles.rename(columns={'batsman': 'player', 'date': 'last_match_date'})
print(f"Batting DNA: {len(profiles)} players, {len(BATTING_DNA)} features")

Batting DNA: 704 players, 22 features


## Cell 3 — Build bowling DNA from master

The batting DNA exists in `player_dna.csv` but there's no equivalent for bowling.
We compute 15 bowling features directly from the ball-by-ball master data.

In [4]:
print("Computing bowling DNA from master data...")
bowl_all = master.copy()
bowl_legal = bowl_all[bowl_all['is_wide'] == 0]

# Minimum 120 balls bowled (~20 overs) to qualify
bowler_balls = bowl_legal.groupby('bowler').size()
qualified_bowlers = bowler_balls[bowler_balls >= 120].index.tolist()
print(f"Qualified bowlers (120+ balls): {len(qualified_bowlers)}")

bowl_q = bowl_all[bowl_all['bowler'].isin(qualified_bowlers)]
bowl_lq = bowl_q[bowl_q['is_wide'] == 0]

# Career aggregates
bowl_career = bowl_lq.groupby('bowler').agg(
    bowl_balls=('bowler', 'count'),
    bowl_runs_conceded=('total_runs', 'sum'),
    bowl_wickets=('is_wicket', 'sum'),
    bowl_dots=('total_runs', lambda x: (x == 0).sum()),
    bowl_matches=('matchId', 'nunique'),
).reset_index()

bowl_career['bowl_economy'] = (bowl_career['bowl_runs_conceded'] / (bowl_career['bowl_balls'] / 6).clip(lower=0.1)).round(2)
bowl_career['bowl_avg'] = (bowl_career['bowl_runs_conceded'] / bowl_career['bowl_wickets'].clip(lower=1)).round(2)
bowl_career['bowl_sr'] = (bowl_career['bowl_balls'] / bowl_career['bowl_wickets'].clip(lower=1)).round(2)
bowl_career['bowl_dot_pct'] = (bowl_career['bowl_dots'] / bowl_career['bowl_balls'].clip(lower=1) * 100).round(1)

# Phase bowling
for phase in ['Powerplay', 'Middle', 'Death']:
    tag = phase.lower()[:2]  # po, mi, de
    p_bowl = bowl_lq[bowl_lq['phase'] == phase]
    ps = p_bowl.groupby('bowler').agg(p_balls=('bowler','count'), p_runs=('total_runs','sum'), p_wkts=('is_wicket','sum')).reset_index()
    ps[f'bowl_economy_{tag}'] = (ps['p_runs'] / (ps['p_balls'] / 6).clip(lower=0.1)).round(2)
    ps[f'bowl_wickets_{tag}'] = (ps['p_wkts'] / ps['p_balls'].clip(lower=1) * 6).round(3)
    bowl_career = bowl_career.merge(ps[['bowler', f'bowl_economy_{tag}', f'bowl_wickets_{tag}']], on='bowler', how='left')

# Best figures + consistency
match_bowl = bowl_q.groupby(['bowler','matchId']).agg(mw=('is_wicket','sum'), mr=('total_runs','sum')).reset_index()
best_fig = match_bowl.groupby('bowler')['mw'].max().reset_index().rename(columns={'mw':'bowl_best_figures'})
bowl_cons = match_bowl.groupby('bowler')['mw'].std().fillna(0).reset_index().rename(columns={'mw':'bowl_consistency'})

# Recent form (last 5 matches)
recent = match_bowl.sort_values('matchId').groupby('bowler').tail(5)
recent_ids = recent.groupby('bowler')['matchId'].apply(set).to_dict()
form_rows = []
for bowler, mids in recent_ids.items():
    r_data = bowl_lq[(bowl_lq['bowler']==bowler) & (bowl_lq['matchId'].isin(mids))]
    r_balls = len(r_data)
    r_runs = r_data['total_runs'].sum()
    r_wkts = r_data['is_wicket'].sum()
    form_rows.append({'bowler': bowler, 'bowl_form_economy': round(r_runs / max(r_balls/6, 0.1), 2), 'bowl_form_wickets': int(r_wkts)})
bowl_form = pd.DataFrame(form_rows)

bowl_career = bowl_career.merge(best_fig, on='bowler', how='left')
bowl_career = bowl_career.merge(bowl_cons, on='bowler', how='left')
bowl_career = bowl_career.merge(bowl_form, on='bowler', how='left')

BOWLING_DNA = [
    'bowl_wickets', 'bowl_economy', 'bowl_avg', 'bowl_sr', 'bowl_dot_pct',
    'bowl_economy_po', 'bowl_economy_mi', 'bowl_economy_de',
    'bowl_wickets_po', 'bowl_wickets_mi', 'bowl_wickets_de',
    'bowl_best_figures', 'bowl_consistency',
    'bowl_form_economy', 'bowl_form_wickets',
]

bowl_dna = bowl_career[['bowler', 'bowl_matches'] + BOWLING_DNA].rename(columns={'bowler': 'player'})
print(f"\nBowling DNA: {len(bowl_dna)} bowlers, {len(BOWLING_DNA)} features")

# Spot check
for name in ['JJ Bumrah', 'YS Chahal', 'Rashid Khan', 'RA Jadeja', 'HH Pandya']:
    row = bowl_dna[bowl_dna['player'] == name]
    if len(row):
        r = row.iloc[0]
        print(f"  {name:18s}  wkts={int(r.bowl_wickets):3d}  eco={r.bowl_economy:.1f}  dot%={r.bowl_dot_pct:.0f}  death_eco={r.bowl_economy_de:.1f}")

Computing bowling DNA from master data...
Qualified bowlers (120+ balls): 332

Bowling DNA: 332 bowlers, 15 features
  JJ Bumrah           wkts=203  eco=7.2  dot%=40  death_eco=8.3
  YS Chahal           wkts=225  eco=7.9  dot%=33  death_eco=9.7
  Rashid Khan         wkts=166  eco=7.1  dot%=36  death_eco=8.4
  RA Jadeja           wkts=178  eco=7.6  dot%=32  death_eco=9.0
  HH Pandya           wkts= 84  eco=9.0  dot%=33  death_eco=12.6


## Cell 4 — Merge batting DNA + bowling DNA + career info + prices

In [5]:
# Merge bowling into profiles
profiles = profiles.merge(bowl_dna, on='player', how='left')

# Merge career info
career_cols = ['player', 'status', 'current_franchise', 'latest_price_cr',
               'debut_season', 'last_season', 'seasons_played']
profiles = profiles.merge(player_career[career_cols], on='player', how='left')
profiles['status'] = profiles['status'].fillna('Retired')

# Player type flags
profiles['is_bowler'] = (profiles['bowl_wickets'].notna()) & (profiles['bowl_wickets'] >= 20)
profiles['is_allrounder'] = (profiles['career_avg'] > 12) & (profiles['is_bowler'])

# Fuzzy-match 2025 auction prices
auc25 = auction_2025.copy()
auc25['price_cr'] = pd.to_numeric(auc25['Sold'], errors='coerce')
auc25_priced = auc25[auc25['price_cr'].notna()][['Players', 'price_cr', 'Type']].copy()
cricsheet_names = profiles['player'].tolist()
name_map = {}
for full_name in auc25_priced['Players'].tolist():
    result = process.extractOne(full_name, cricsheet_names, scorer=fuzz.WRatio, score_cutoff=80)
    if result: name_map[full_name] = result[0]
auc25_priced['cricsheet_name'] = auc25_priced['Players'].map(name_map)
auc25_matched = auc25_priced.dropna(subset=['cricsheet_name'])
price_map = dict(zip(auc25_matched['cricsheet_name'], auc25_matched['price_cr']))
type_map = dict(zip(auc25_matched['cricsheet_name'], auc25_matched['Type']))

profiles['price_cr'] = pd.to_numeric(profiles['latest_price_cr'], errors='coerce')
for player, price in price_map.items():
    mask = profiles['player'] == player
    if mask.any() and pd.isna(profiles.loc[mask, 'price_cr'].iloc[0]):
        profiles.loc[mask, 'price_cr'] = price
profiles['role_type'] = profiles['player'].map(type_map)

print(f"Total profiles: {len(profiles)}")
print(f"  Pure batters: {(~profiles['is_bowler']).sum()}")
print(f"  Pure bowlers: {(profiles['is_bowler'] & ~profiles['is_allrounder']).sum()}")
print(f"  All-rounders: {profiles['is_allrounder'].sum()}")
print(f"  With price: {profiles['price_cr'].notna().sum()}")

Total profiles: 704
  Pure batters: 532
  Pure bowlers: 135
  All-rounders: 37
  With price: 165


## Cell 5 — Performance rating (batting + bowling combined)

Batters are scored on batting metrics. Bowlers are scored on bowling metrics (economy inverted so lower = better).
All-rounders get a weighted blend of both.

In [6]:
# Batting performance
profiles['bat_performance'] = (
    profiles['career_avg'] * 0.25 + profiles['career_sr'] * 0.15 +
    profiles['form_score'] * 0.20 + profiles['boundary_rate'] * 100 * 0.10 +
    profiles['avg_sr_Death'] * 0.10 + profiles['high_pressure_avg'] * 0.10 +
    profiles['peak_score'] * 0.10
)

# Bowling performance (lower economy = better, so we invert)
profiles['bowl_performance'] = 0.0
bm = profiles['bowl_wickets'].notna() & (profiles['bowl_wickets'] > 0)
profiles.loc[bm, 'bowl_performance'] = (
    profiles.loc[bm, 'bowl_wickets'] * 0.20 +
    (12 - profiles.loc[bm, 'bowl_economy'].clip(upper=12)) * 10 * 0.20 +
    profiles.loc[bm, 'bowl_dot_pct'] * 0.15 +
    (12 - profiles.loc[bm, 'bowl_economy_de'].clip(upper=12).fillna(8)) * 10 * 0.15 +
    profiles.loc[bm, 'bowl_best_figures'].fillna(0) * 8 * 0.10 +
    profiles.loc[bm, 'bowl_wickets_de'].fillna(0) * 100 * 0.10 +
    (30 - profiles.loc[bm, 'bowl_sr'].clip(upper=30)) * 2 * 0.10
)

# Combined: batter=100% bat, bowler=100% bowl, AR=60/40 weighted to stronger skill
def combined_perf(row):
    bat, bowl = row['bat_performance'], row['bowl_performance']
    if row.get('is_allrounder', False):
        return max(bat, bowl) * 0.6 + min(bat, bowl) * 0.4
    elif row.get('is_bowler', False):
        return bowl
    return bat

profiles['performance_score'] = profiles.apply(combined_perf, axis=1)

pmin, pmax = profiles['performance_score'].quantile(0.05), profiles['performance_score'].quantile(0.95)
profiles['performance_rating'] = ((profiles['performance_score'] - pmin) / (pmax - pmin) * 100).clip(0, 100).round(1)

mask = profiles['price_cr'].notna() & (profiles['price_cr'] > 0)
profiles.loc[mask, 'value_score'] = (profiles.loc[mask, 'performance_rating'] / profiles.loc[mask, 'price_cr']).round(2)

print("Top 10 bowlers by performance:")
bowlers = profiles[profiles['is_bowler'] & ~profiles['is_allrounder']]
print(bowlers.nlargest(10, 'performance_rating')[['player','performance_rating','bowl_wickets','bowl_economy','bowl_dot_pct']].to_string(index=False))

print("\nTop 10 all-rounders:")
ars = profiles[profiles['is_allrounder']]
print(ars.nlargest(10, 'performance_rating')[['player','performance_rating','career_avg','bowl_wickets','bowl_economy']].to_string(index=False))

Top 10 bowlers by performance:
         player  performance_rating  bowl_wickets  bowl_economy  bowl_dot_pct
       A Mishra               100.0         182.0          7.31          34.9
        B Kumar               100.0         213.0          7.60          41.3
       HV Patel               100.0         168.0          8.58          32.4
Harbhajan Singh               100.0         157.0          7.02          36.9
      JJ Bumrah               100.0         203.0          7.18          39.9
      PP Chawla               100.0         198.0          7.94          34.4
       R Ashwin               100.0         201.0          7.04          34.1
    Rashid Khan               100.0         166.0          7.14          35.9
     SL Malinga               100.0         186.0          7.02          40.2
      YS Chahal               100.0         225.0          7.87          33.3

Top 10 all-rounders:
    player  performance_rating  career_avg  bowl_wickets  bowl_economy
 SP Narine        

## Cell 6 — Scouting score (5 components)

In [7]:
def normalize_col(s):
    mn, mx = s.quantile(0.05), s.quantile(0.95)
    if mx == mn: return pd.Series(50.0, index=s.index)
    return ((s - mn) / (mx - mn) * 100).clip(0, 100)

profiles['form_rating'] = normalize_col(profiles['form_score']).round(1)
profiles['consistency_rating'] = (100 - normalize_col(profiles['consistency'])).round(1)
impact_raw = profiles['peak_score'] * 0.6 + profiles['boundary_rate'] * 100 * 0.4
profiles['impact_rating'] = normalize_col(impact_raw).round(1)
profiles['pressure_rating'] = normalize_col(profiles['high_pressure_avg']).round(1)

profiles['scouting_score'] = (
    profiles['performance_rating'] * 0.30 + profiles['form_rating'] * 0.25 +
    profiles['consistency_rating'] * 0.15 + profiles['impact_rating'] * 0.15 +
    profiles['pressure_rating'] * 0.15
).round(1)

active = profiles[profiles['status'].isin(['Active', 'Unsold'])]
print("Top 15 by scouting score (Active/Unsold):")
print(active.nlargest(15, 'scouting_score')[['player','scouting_score','performance_rating','playstyle' if 'playstyle' in active.columns else 'is_bowler']].to_string(index=False))

Top 15 by scouting score (Active/Unsold):
         player  scouting_score  performance_rating  is_bowler
      H Klaasen            84.8               100.0      False
     JC Buttler            84.8               100.0      False
      RG Sharma            83.5                94.7      False
       SA Yadav            83.5               100.0      False
   Shubman Gill            83.1               100.0      False
       KL Rahul            82.9               100.0      False
       N Pooran            82.8                99.7      False
      SV Samson            82.8                94.8      False
   Ishan Kishan            82.6                91.6      False
        V Kohli            82.1               100.0      False
        SS Iyer            81.6               100.0      False
Abhishek Sharma            80.7               100.0      False
        R Parag            80.4                89.6      False
    YBK Jaiswal            80.4               100.0      False
      JM Shar

## Cell 7 — Playstyle classification (11 categories)

**Batting:** Aggressive, Anchor, Finisher, Impact, Utility
**Bowling:** Death Specialist, Restrictive Bowler, Wicket-taker, Support Bowler
**All-round:** Batting Allrounder, Bowling Allrounder

In [8]:
def classify_playstyle(row):
    sr = row.get('career_sr', 0) or 0
    avg = row.get('career_avg', 0) or 0
    br = row.get('boundary_rate', 0) or 0
    death_sr = row.get('avg_sr_Death', 0) or 0
    is_bowler = row.get('is_bowler', False)
    is_ar = row.get('is_allrounder', False)
    bowl_eco = row.get('bowl_economy', 99) or 99
    bowl_eco_de = row.get('bowl_economy_de', 99) or 99
    bowl_dot = row.get('bowl_dot_pct', 0) or 0
    
    if is_ar:
        return 'Batting Allrounder' if avg > 20 or sr > 135 else 'Bowling Allrounder'
    if is_bowler:
        if bowl_eco_de < 8.0 and bowl_eco < 7.5: return 'Death Specialist'
        elif bowl_eco < 7.0 and bowl_dot > 45: return 'Restrictive Bowler'
        elif bowl_eco < 8.5: return 'Wicket-taker'
        else: return 'Support Bowler'
    if sr > 145 and br > 0.17: return 'Aggressive'
    elif avg > 25 and sr < 125: return 'Anchor'
    elif death_sr > 145 and sr > 130: return 'Finisher'
    elif avg > 20 and sr > 125: return 'Impact'
    return 'Utility'

profiles['playstyle'] = profiles.apply(classify_playstyle, axis=1)
print("Playstyle distribution:")
print(profiles['playstyle'].value_counts().to_string())
print()
for style in sorted(profiles['playstyle'].unique()):
    top = profiles[(profiles['playstyle']==style) & profiles['status'].isin(['Active','Unsold'])].nlargest(3, 'scouting_score' if 'scouting_score' in profiles.columns else 'performance_rating')
    if len(top): print(f"  {style:22s}: {', '.join(top['player'].tolist())}")

Playstyle distribution:
playstyle
Utility               400
Wicket-taker           87
Anchor                 44
Support Bowler         43
Aggressive             38
Finisher               26
Impact                 24
Batting Allrounder     19
Bowling Allrounder     18
Death Specialist        4
Restrictive Bowler      1

  Aggressive            : H Klaasen, N Pooran, TH David
  Anchor                : RG Sharma, Shubman Gill, SV Samson
  Batting Allrounder    : AD Russell, MR Marsh, HH Pandya
  Bowling Allrounder    : RA Jadeja, SP Narine, SM Curran
  Finisher              : SA Yadav, YBK Jaiswal, JM Sharma
  Impact                : JC Buttler, KL Rahul, Abhishek Sharma
  Support Bowler        : HV Patel, SN Thakur, K Rabada
  Utility               : MA Agarwal, Dhruv Jurel, R Ravindra
  Wicket-taker          : Rashid Khan, R Ashwin, UT Yadav


## Cell 8 — Similarity engine (37 features: 22 batting + 15 bowling)

Bowlers are matched with bowlers, batters with batters, all-rounders with all-rounders.
The combined 37-dimensional DNA captures the full player profile.

In [9]:
ALL_DNA = BATTING_DNA + BOWLING_DNA
dna_matrix = profiles[ALL_DNA].fillna(0).values
scaler = MinMaxScaler()
dna_scaled = scaler.fit_transform(dna_matrix)
sim_matrix = cosine_similarity(dna_scaled)
player_index = {name: i for i, name in enumerate(profiles['player'].values)}

def find_similar_players(player_name, top_n=5, status_filter=None, budget_max=None, same_type=True):
    if player_name not in player_index:
        print(f"'{player_name}' not found"); return pd.DataFrame()
    idx = player_index[player_name]
    source = profiles.iloc[idx]
    results = profiles[['player','career_avg','career_sr','bowl_wickets','bowl_economy',
                        'status','current_franchise','price_cr','scouting_score','playstyle',
                        'is_bowler','is_allrounder']].copy()
    results['similarity'] = (sim_matrix[idx] * 100).round(1)
    results = results[results['player'] != player_name]
    if same_type:
        if source.get('is_allrounder', False): results = results[results['is_allrounder']==True]
        elif source.get('is_bowler', False): results = results[results['is_bowler']==True]
        else: results = results[(results['is_bowler']==False) & (results['is_allrounder']==False)]
    if status_filter: results = results[results['status']==status_filter]
    if budget_max is not None: results = results[(results['price_cr'].isna())|(results['price_cr']<=budget_max)]
    return results.nlargest(top_n, 'similarity')

print("Bowlers similar to JJ Bumrah:")
s1 = find_similar_players('JJ Bumrah', top_n=8, status_filter='Active')
if len(s1): print(s1[['player','similarity','bowl_wickets','bowl_economy','playstyle']].to_string(index=False))

print("\nAll-rounders similar to HH Pandya:")
s2 = find_similar_players('HH Pandya', top_n=8)
if len(s2): print(s2[['player','similarity','career_avg','bowl_wickets','bowl_economy']].to_string(index=False))

print("\nBatters similar to V Kohli:")
s3 = find_similar_players('V Kohli', top_n=8, status_filter='Active')
if len(s3): print(s3[['player','similarity','career_avg','career_sr','playstyle']].to_string(index=False))

Bowlers similar to JJ Bumrah:
           player  similarity  bowl_wickets  bowl_economy      playstyle
         TA Boult        97.4         151.0          8.22   Wicket-taker
         CV Varun        96.3         101.0          7.57   Wicket-taker
         R Ashwin        95.4         201.0          7.04   Wicket-taker
Mustafizur Rahman        95.3          74.0          7.99   Wicket-taker
   Mohammed Siraj        95.2         112.0          8.49   Wicket-taker
         K Rabada        94.4         128.0          8.52 Support Bowler
        SN Thakur        94.3         115.0          9.09 Support Bowler
        MM Sharma        93.8         149.0          8.65 Support Bowler

All-rounders similar to HH Pandya:
    player  similarity  career_avg  bowl_wickets  bowl_economy
AD Russell        99.0   23.495575         133.0          9.34
MP Stoinis        98.7   20.612245          46.0          9.70
   J Botha        97.7   15.074074          27.0          6.91
 RA Jadeja        97.6   

## Cell 9 — Recommendation engine

Now supports all player types: batters, bowlers, and all-rounders.

In [10]:
TEAM_FULL = {
    'RCB':'Royal Challengers Bengaluru','MI':'Mumbai Indians','CSK':'Chennai Super Kings',
    'KKR':'Kolkata Knight Riders','DC':'Delhi Capitals','SRH':'Sunrisers Hyderabad',
    'PBKS':'Punjab Kings','RR':'Rajasthan Royals','GT':'Gujarat Titans','LSG':'Lucknow Super Giants',
}

def recommend_players(team=None, playstyle=None, budget_max=None, top_n=10):
    r = profiles[profiles['status'].isin(['Active','Unsold'])].copy()
    if team:
        full = TEAM_FULL.get(team, team)
        r = r[r['current_franchise'] != full]
    if playstyle: r = r[r['playstyle']==playstyle]
    if budget_max: r = r[(r['price_cr'].isna())|(r['price_cr']<=budget_max)]
    r = r[r['scouting_score'] > 20]
    return r.nlargest(top_n, 'scouting_score')[
        ['player','scouting_score','performance_rating','playstyle','career_avg','career_sr',
         'bowl_wickets','bowl_economy','price_cr','value_score','status','current_franchise']]

print("--- Death Specialist bowlers ---")
print(recommend_players(playstyle='Death Specialist', top_n=8).to_string(index=False))

print("\n--- Wicket-takers under 10 Cr ---")
print(recommend_players(playstyle='Wicket-taker', budget_max=10, top_n=8).to_string(index=False))

print("\n--- Bowling Allrounders for CSK ---")
print(recommend_players(team='CSK', playstyle='Bowling Allrounder', top_n=8).to_string(index=False))

print("\n--- Aggressive batters under 8 Cr ---")
print(recommend_players(playstyle='Aggressive', budget_max=8, top_n=8).to_string(index=False))

--- Death Specialist bowlers ---
Empty DataFrame
Columns: [player, scouting_score, performance_rating, playstyle, career_avg, career_sr, bowl_wickets, bowl_economy, price_cr, value_score, status, current_franchise]
Index: []

--- Wicket-takers under 10 Cr ---
           player  scouting_score  performance_rating    playstyle  career_avg  career_sr  bowl_wickets  bowl_economy  price_cr  value_score status   current_franchise
         R Ashwin            63.6               100.0 Wicket-taker    8.913043 105.955978         201.0          7.04      9.75        10.26 Active Chennai Super Kings
         UT Yadav            57.5                93.1 Wicket-taker    4.382979  93.763191         163.0          8.42       NaN          NaN Unsold              Unsold
   Mohammed Shami            54.8                92.0 Wicket-taker    3.000000  70.512692         150.0          8.46     10.00         9.20 Active Sunrisers Hyderabad
Washington Sundar            54.5                47.0 Wicket-taker  

## Cell 10 — Save all outputs

In [11]:
save_cols = ['player','last_match_date','matches_played'] + BATTING_DNA + BOWLING_DNA + [
    'status','current_franchise','price_cr','debut_season','last_season','seasons_played',
    'role_type','is_bowler','is_allrounder',
    'performance_rating','form_rating','consistency_rating','impact_rating','pressure_rating',
    'scouting_score','value_score','playstyle'
]
save_cols = [c for c in save_cols if c in profiles.columns]

profiles[save_cols].to_csv('../data/scouting_profiles.csv', index=False)
np.save('../data/dna_matrix_scaled.npy', dna_scaled)
np.save('../data/player_names_index.npy', profiles['player'].values)

print(f"Saved scouting_profiles.csv: {len(profiles)} players, {len(save_cols)} columns")
print(f"  Pure batters: {(~profiles['is_bowler'] & ~profiles['is_allrounder']).sum()}")
print(f"  Pure bowlers: {(profiles['is_bowler'] & ~profiles['is_allrounder']).sum()}")
print(f"  All-rounders: {profiles['is_allrounder'].sum()}")
print(f"  DNA matrix: {dna_scaled.shape} (22 batting + 15 bowling = 37 dimensions)")
print("\n✅ Module 2 complete!")

Saved scouting_profiles.csv: 704 players, 57 columns
  Pure batters: 532
  Pure bowlers: 135
  All-rounders: 37
  DNA matrix: (704, 37) (22 batting + 15 bowling = 37 dimensions)

✅ Module 2 complete!
